# `mock-fastq-generator`: Quality Decay Models Demo

This notebook demonstrates how to generate synthetic FASTQ datasets with three distinct scientific quality decay models:
1. **Parametric Gaussian Decay**
2. **Exponential 3' Read Degradation**
3. **Sigmoidal Drop with NovaSeq Quality Binning & Homopolymer Penalty**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mock_fastq_generator.core import assemble_sequences

## 1. Parametric Gaussian Decay Model

The default Gaussian model allows targeted placement of peak quality at index $C$ with standard deviation $\sigma$ and rich experimental noise.

In [ ]:
template_seq = "GCCGGCCATGGCGAAAAAAATTTTTTTCCCCCCGGGGGGGGATCGATCGATCG"
res_gauss = assemble_sequences(
    template_sequence=template_seq,
    total_length=500,
    number_of_sequences=1,
    center=160,
    min_val=40, # Phred 7
    max_val=73, # Phred 40
    std_dev=90,
    noise_level=0.55,
    decay_model="gaussian"
)
scores_gauss = [ord(c) - 33 for c in res_gauss[0][3]]

plt.figure(figsize=(10, 4.5))
plt.plot(scores_gauss, label="Experimental Read Profile", color="#1f77b4", alpha=0.88)
plt.title("A) Parametric Gaussian Quality Decay", fontsize=14, fontweight="bold")
plt.xlabel("Base Position (Index)", fontsize=12)
plt.ylabel("Phred Quality Score (Q)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=11)
plt.show()

## 2. Exponential 3' Read Degradation Model

Models non-linear experimental signal degradation towards read 3' ends, with gentle early-cycle decay and accelerating loss near the read terminus.

In [ ]:
res_exp = assemble_sequences(
    template_sequence=template_seq,
    total_length=500,
    number_of_sequences=1,
    min_val=40,
    max_val=73,
    decay_model="exponential",
    decay_rate=0.0008,
    noise_level=0.50
)
scores_exp = [ord(c) - 33 for c in res_exp[0][3]]

plt.figure(figsize=(10, 4.5))
plt.plot(scores_exp, label="Experimental Read Profile", color="#ff7f0e", alpha=0.88)
plt.title("B) Exponential 3' Read Quality Degradation", fontsize=14, fontweight="bold")
plt.xlabel("Base Position (Index)", fontsize=12)
plt.ylabel("Phred Quality Score (Q)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=11)
plt.show()

## 3. Sigmoidal Drop with NovaSeq Binning & Homopolymer Penalty

Simulates discrete 4-state Illumina NovaSeq quality score binning (Q2, Q11, Q25, Q37) and penalty drops inside homopolymer runs (>4 identical consecutive bases).

In [ ]:
res_sig = assemble_sequences(
    template_sequence=template_seq,
    total_length=500,
    number_of_sequences=1,
    center=260,
    min_val=40,
    max_val=73,
    decay_model="sigmoidal",
    decay_rate=0.020,
    binned_quality=True,
    homopolymer_penalty=True,
    noise_level=0.15
)
scores_sig = [ord(c) - 33 for c in res_sig[0][3]]

plt.figure(figsize=(10, 4.5))
plt.plot(scores_sig, drawstyle="steps-mid", label="Binned + Penalty", color="#2ca02c")
plt.title("C) Sigmoidal Drop + NovaSeq Binning & Homopolymer Penalty", fontsize=14, fontweight="bold")
plt.xlabel("Base Position (Index)", fontsize=12)
plt.ylabel("Phred Quality Score (Q)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=11)
plt.show()

## 4. Consolidated 3-Panel Overview Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), sharey=True)

# Panel A
axes[0].plot(scores_gauss, color="#1f77b4", alpha=0.88, label="Gaussian")
axes[0].set_title("A) Parametric Gaussian", fontweight="bold", fontsize=16)
axes[0].set_xlabel("Base Position (Index)", fontsize=13)
axes[0].set_ylabel("Phred Quality Score (Q)", fontsize=13)
axes[0].grid(True, linestyle="--", alpha=0.35)

# Panel B
axes[1].plot(scores_exp, color="#ff7f0e", alpha=0.88, label="Exponential")
axes[1].set_title("B) Exponential 3' Drop", fontweight="bold", fontsize=16)
axes[1].set_xlabel("Base Position (Index)", fontsize=13)
axes[1].grid(True, linestyle="--", alpha=0.35)

# Panel C
axes[2].plot(scores_sig, color="#2ca02c", drawstyle="steps-mid", label="Binned + Penalty")
axes[2].set_title("C) Sigmoidal + Binning", fontweight="bold", fontsize=16)
axes[2].set_xlabel("Base Position (Index)", fontsize=13)
axes[2].grid(True, linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig("../img/decay_models_three_panel.png", dpi=300)
plt.show()

### Data Analysis Key Findings
- **Parametric Gaussian Decay**: Ideal for targeted fault injection with controllable peak center, standard deviation, and realistic per-cycle noise.
- **Exponential 3' Decay**: Accurately models non-linear signal degradation towards read ends with gentle early decay and accelerating 3' quality loss.
- **NovaSeq Binning & Homopolymer Penalty**: Replicates modern 4-state platform binning ($Q2, Q11, Q25, Q37$) and homopolymer quality dropoffs for testing preprocessing algorithms.